# UNIFICAR DATOS DE VARIOS CSV

Con esto conseguimos tener todos los datos de Data Science y Full Stack en las mismas tablas

In [10]:
import pandas as pd

In [26]:
import psycopg2

### DATAFRAME CAMPUS, MODALIDAD Y ROL

In [11]:
df_campus = pd.DataFrame({
    'cod_campus': [1, 2],
    'nombre': ['Madrid', 'Valencia'],
})

df_modalidad = pd.DataFrame({
    'cod_modalidad': [1, 2],
    'nombre': ['Presencial', 'Online'],
})

df_rol = pd.DataFrame({
    'cod_rol': [1, 2],
    'nombre': ['TA', 'LI'],
})

# Diccionario que relaciona cada código con su nombre
# zip: Empareja elemento a elemento ambas columnas en orden
diccionario_campus = dict(zip(df_campus['nombre'], df_campus['cod_campus']))
diccionario_modalidad = dict(zip(df_modalidad['nombre'], df_modalidad['cod_modalidad']))
diccionario_rol = dict(zip(df_rol['nombre'], df_rol['cod_rol']))

In [12]:
diccionario_campus

{'Madrid': 1, 'Valencia': 2}

In [13]:
diccionario_modalidad

{'Presencial': 1, 'Online': 2}

In [14]:
diccionario_rol

{'TA': 1, 'LI': 2}

In [15]:
# CLASES Y CLAUSTRO

clases_config = [
    ('data/clase_1.csv', 'DS'),
    ('data/clase_2.csv', 'DS'),
    ('data/clase_3.csv', 'FS'),
    ('data/clase_4.csv', 'FS'),
]

dfs_clases = []
proyectos_vertical = {}

for archivo, vertical in clases_config:
    df_clase = pd.read_csv(archivo, sep=';')
    if 'Promoción' in df_clase.columns and 'Promocion' not in df_clase.columns:
        df_clase = df_clase.rename(columns={'Promoción':'Promocion'})
    cols_proy = [c for c in df_clase.columns if c.startswith('Proyecto_')]
    for col_p in cols_proy:
        proyectos_vertical[col_p] = vertical
    dfs_clases.append(df_clase)


df_all_alumnos = pd.concat(dfs_clases, ignore_index=True)
df_claustro = pd.read_csv('data/claustro.csv', sep=';')

In [16]:
df_all_alumnos

,Nombre,Email,Promocion,Fecha_comienzo,Campus,Proyecto_HLF,Proyecto_EDA,Proyecto_BBDD,Proyecto_ML,Proyecto_Deployment,Proyecto_WebDev,Proyecto_FrontEnd,Proyecto_Backend,Proyecto_React,Proyecto_FullSatck
0,Jafet Casals,Jafet_Casals@gmail.com,Septiembre,18/09/2023,Madrid,Apto,No Apto,Apto,Apto,Apto,NaN,NaN,NaN,NaN,NaN
1,Jorge Manzanares,Jorge_Manzanares@gmail.com,Septiembre,18/09/2023,Madrid,Apto,No Apto,Apto,Apto,Apto,NaN,NaN,NaN,NaN,NaN
2,Onofre Adadia,Onofre_Adadia@gmail.com,Septiembre,18/09/2023,Madrid,Apto,Apto,Apto,No Apto,Apto,NaN,NaN,NaN,NaN,NaN
3,Merche Prada,Merche_Prada@gmail.com,Septiembre,18/09/2023,Madrid,Apto,No Apto,No Apto,Apto,No Apto,NaN,NaN,NaN,NaN,NaN
4,Pilar Abella,Pilar_Abella@gmail.com,Septiembre,18/09/2023,Madrid,Apto,No Apto,Apto,Apto,Apto,NaN,NaN,NaN,NaN,NaN
5,Leoncio Tena,Leoncio_Tena@gmail.com,Septiembre,18/09/2023,Madrid,Apto,Apto,Apto,Apto,Apto,NaN,NaN,NaN,NaN,NaN
6,Odalys Torrijos,Odalys_Torrijos@gmail.com,Septiembre,18/09/2023,Madrid,No Apto,Apto,Apto,Apto,Apto,NaN,NaN,NaN,NaN,NaN
7,Eduardo Caparrós,Eduardo_Caparrós@gmail.com,Septiembre,18/09/2023,Madrid,No Apto,Apto,Apto,Apto,Apto,NaN,NaN,NaN,NaN,NaN
8,Ignacio Goicoechea,Ignacio_Goicoechea@gmail.com,Septiembre,18/09/2023,Madrid,Apto,Apto,Apto,No Apto,Apto,NaN,NaN,NaN,NaN,NaN
9,Clementina Santos,Clementina_Santos@gmail.com,Septiembre,18/09/2023,Madrid,Apto,No Apto,Apto,Apto,Apto,NaN,NaN,NaN,NaN,NaN


In [17]:
df_claustro

,Nombre,Rol,Vertical,Promocion,Campus,Modalidad
0,Noa Yáñez,TA,DS,Septiembre,Madrid,Presencial
1,Saturnina Benitez,TA,DS,Septiembre,Madrid,Presencial
2,Anna Feliu,TA,FS,Septiembre,Madrid,Presencial
3,Rosalva Ayuso,TA,FS,Septiembre,Valencia,Presencial
4,Ana Sofía Ferrer,TA,FS,Febrero,Valencia,Presencial
5,Angélica Corral,TA,FS,Febrero,Madrid,Presencial
6,Ariel Lledó,TA,DS,Septiembre,Madrid,Presencial
7,Mario Prats,LI,FS,Febrero,Valencia,Online
8,Luis Ángel Suárez,LI,FS,Septiembre,Madrid,Online
9,María Dolores Diaz,LI,DS,Septiembre,Madrid,Online


### DATAFRAME PROMOCIÓN

map(): Se aplica sobre una serie para sustituir o transformar cada valor, de forma elemento a elemento.  
Con un diccionario (dict): Busca cada valor de la Serie como clave en el diccionario y devuelve su valor asociado.

In [23]:
proms = (
    df_all_alumnos[['Promocion', 'Fecha_comienzo', 'Campus']].drop_duplicates().reset_index(drop=True)
)
proms['cod_promocion'] = range(1, len(proms)+1)
proms['fecha_comienzo'] = proms['Fecha_comienzo']
proms['cod_campus'] = proms['Campus'].map(diccionario_campus)

df_promocion = proms[['cod_promocion', 'fecha_comienzo', 'cod_campus']]

diccionario_promo_campus = {
    (row['Promocion'], row['Campus']): row['cod_promocion']
    for _, row in proms.iterrows()
}
diccionario_promo_nombre = proms.groupby('Promocion')['cod_promocion'].first().to_dict()

In [25]:
diccionario_promo_nombre

{'Febrero': 2, 'Septiembre': 1}

In [24]:
df_promocion

,cod_promocion,fecha_comienzo,cod_campus
0,1,18/09/2023,1
1,2,12/02/2024,1
2,3,12/02/2024,2


### DATAFRAME PROYECTO

### DATAFRAME PROFESOR